# <font style="font-family:roboto;color:#455e6c"> Calculation of phase diagrams </font>  

<div class="admonition note" name="html-admonition" style="background:#e3f2fd; padding: 10px">
<font style="font-family:roboto;color:#455e6c"> <b> DPG Tutorial: Automated Workflows and Machine Learning for Materials Science Simulations </b> </font> </br>
<font style="font-family:roboto;color:#455e6c"> 16 March 2025 </font> </br> </br>
Marvin Poul, Sarath Menon, Haitham Gaafer, Jörg Neugebauer </br>
<i> Max Planck Institute for Sustainable Materials </i></br>
</br>
Minaam Qamar, Ralf Drautz </br>
<i> Ruhr-Universität Bochum </i></br>
</br>
Tilmann Hickel </br>
<i> Bundesanstalt für Materialforschung und -prüfung </i></br>
</div>

### References

- [Menon, S. et al. From electrons to phase diagrams with machine learning potentials using pyiron based automated workflows. npj Comput Mater 10, 261 (2024)](https://www.nature.com/articles/s41524-024-01441-0)
- [Menon, S., Lysogorskiy, Y., Rogal, J. & Drautz, R. Automated free-energy calculation from atomistic simulations. Phys. Rev. Materials 5, 103801 (2021)](https://link.aps.org/doi/10.1103/PhysRevMaterials.5.103801)
- [Poul, M., Huber, L. & Neugebauer, J. Automated Generation of Structure Datasets for Machine Learning Potentials and Alloys. Preprint](https://doi.org/10.21203/rs.3.rs-4732459/v1) (2024)


In this notebook, we will use the interatomic potentials for calculation of thermodynamic properties such as Helmholtz and Gibbs free energies, which in turn can be used for the calculation of phase diagrams. We will discuss [calphy](https://calphy.org/), the tool for automated calculation of free energies, and the methology involved. Furthermore, we will also discuss [Landau](https://github.com/eisenforschung/landau) which can be used to arrive at phase diagrams from free energies.

## <font style="font-family:roboto;color:#455e6c"> A simple phase diagram </font> 

<img src="img/phase_dia_1.png" width="40%">

Phase diagrams provide a wealth of information such as: coexisting lines, melting temperature, phase stability, nucleation mechanism.

## <font style="font-family:roboto;color:#455e6c"> Calculation of phase diagrams: the essential ingredients </font> 

<img src="img/cimg4.png" width="30%">

Phase diagrams can be evaluated from free energy diagrams. The required input are:
- $G(P, T)$ for unary systems
- $G(x, T)$ for binary systems

To calculate this, we employ thermodynamic integration, in conjuction with a number of methodological improvements.

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Theory behind free energy calculation </b></p>
Please see the end of this notebook, or  <a href="https://journals.aps.org/prmaterials/abstract/10.1103/PhysRevMaterials.5.103801">this publication</a> for a longer discussion on the theory and algorithms.
</div>

Due to our limited time, we will show the calculation of melting temperature, the end point on the phase diagram. We leverage a software tool for calculation of free energies, [calphy](https://calphy.org) along with pyiron for efficient and automated free energy calculations. For a detailed tutorial, see [here](https://workshop.pyiron.org/potentials-workshop-2022/phase_diagram/Intro.html).

In [1]:
from pyiron_core import Workflow, PyironFlow, as_function_node, as_inp_dataclass_node

/cmmc/ptmp/pyironhb/mambaforge/envs/pyiron_mpie_cmti_2026-01-26/lib/python3.12/site-packages/nglview/__init__.py:12: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## <font style="font-family:roboto;color:#455e6c"> Calculate free energy with temperature for solid</font> 

By now we have seen how the GUI works. We try to calculate the free energy of an Al FCC structure.

Helpful nodes:
- `atomistic -> structure -> build -> Bulk` to create a structure. Make sure to check `cubic` box.
- `atomistic -> structure -> transform -> Repeat` to create a supercell. A 3x3x3 cell should work for this example.
- `atomistic -> thermodynamics -> calphy -> InputClass` to set inputs for free energy calculation.
- `atomistic -> thermodynamics -> calphy -> SolidFreeEnergyWithTemp` for calculating the free energy. A good potential which is fast to try out this calculation would be `2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1`
- `atomistic -> thermodynamics -> calphy -> PlotFreeEnergy` for a plot.

In [2]:
from dataclasses import dataclass
from typing import Optional

In [3]:
@as_function_node
def ListPotentials(structure):
    from pyiron_atomistics.lammps.potential import list_potentials as lp

    potentials = lp(structure)
    return potentials

In [4]:
@as_inp_dataclass_node
@dataclass
class MD:
    """
    Molecular dynamics parameters.

    Attributes:
    -----------
    timestep: float
        https://calphy.org/en/latest/inputfile.html#timestep
    n_small_steps: int
        https://calphy.org/en/latest/inputfile.html#n-small-steps
    n_every_steps: int
        https://calphy.org/en/latest/inputfile.html#n-every-steps
    n_repeat_steps: int
        https://calphy.org/en/latest/inputfile.html#n-repeat-steps
    n_cycles: int
        https://calphy.org/en/latest/inputfile.html#n-cycles
    thermostat_damping: float
        https://calphy.org/en/latest/inputfile.html#thermostat-damping
    barostat_damping: float
        https://calphy.org/en/latest/inputfile.html#barostat-damping
    """
    timestep: float = 0.001
    n_small_steps: int = 10000
    n_every_steps: int = 10
    n_repeat_steps: int = 10
    n_cycles: int = 100
    thermostat_damping: float = 0.5
    barostat_damping: float = 0.1

In [5]:
@as_inp_dataclass_node
@dataclass
class Tolerance:
    """
    Tolerance parameters.

    Attributes:
    -----------
    spring_constant: float
        https://calphy.org/en/latest/inputfile.html#tol-spring-constant
    solid_fraction: float
        https://calphy.org/en/latest/inputfile.html#tol-solid-fraction
    liquid_fraction: float
        https://calphy.org/en/latest/inputfile.html#tol-liquid-fraction
    pressure: float
        https://calphy.org/en/latest/inputfile.html#tol-pressure
    """
    spring_constant: float = 0.01
    solid_fraction: float = 0.7
    liquid_fraction: float = 0.05
    pressure: float = 1.0

In [6]:
@as_inp_dataclass_node
@dataclass
class NoseHoover:
    """
    Nose-Hoover parameters.

    Attributes:
    -----------
    thermostat_damping: float
        https://calphy.org/en/latest/inputfile.html#nose-hoover-thermostat-damping
    barostat_damping: float
        https://calphy.org/en/latest/inputfile.html#nose-hoover-barostat-damping
    """
    thermostat_damping: float = 0.1
    barostat_damping: float = 0.1

In [7]:
@as_inp_dataclass_node
@dataclass
class Berendsen:
    """
    Berendsen parameters.

    Attributes:
    -----------
    thermostat_damping: float
        https://calphy.org/en/latest/inputfile.html#berendsen-thermostat-damping
    barostat_damping: float
        https://calphy.org/en/latest/inputfile.html#berendsen-barostat-damping
    """
    thermostat_damping: float = 100.0
    barostat_damping: float = 100.0

In [8]:
@as_inp_dataclass_node
@dataclass
class InputClass:
    """
    Input parameters for calphy calculations.

    Attributes:
    -----------
    md: MD
        Molecular dynamics parameters.
    tolerance: Tolerance
        Tolerance parameters.
    nose_hoover: NoseHoover
        Nose-Hoover parameters.
    berendsen: Berendsen
        Berendsen parameters.
    queue: Queue
        Queue parameters.
    pressure: int
        https://calphy.org/en/latest/inputfile.html#pressure
    temperature: int
        https://calphy.org/en/latest/inputfile.html#temperature
    npt: bool
        https://calphy.org/en/latest/inputfile.html#npt
    n_equilibration_steps: int
        https://calphy.org/en/latest/inputfile.html#n-equilibration-steps
    n_switching_steps: int
        https://calphy.org/en/latest/inputfile.html#n-switching-steps
    n_print_steps: int
        https://calphy.org/en/latest/inputfile.html#n-print-steps
    n_iterations: int
        https://calphy.org/en/latest/inputfile.html#n-iterations
    equilibration_control: str
        https://calphy.org/en/latest/inputfile.html#equilibration-control
    melting_cycle: bool
        https://calphy.org/en/latest/inputfile.html#melting-cycle
    spring_constants: Optional[float]
        https://calphy.org/en/latest/inputfile.html#spring-constants        
    """
    md: Optional[MD] = None 
    tolerance: Optional[Tolerance] = None
    nose_hoover: Optional[NoseHoover] = None
    berendsen: Optional[Berendsen] = None
    pressure: int = 0
    temperature: int = 300
    temperature_stop: int = 600
    npt: bool = True
    n_equilibration_steps: int = 2500
    n_switching_steps: int = 2500
    n_print_steps: int = 1000
    n_iterations: int = 1
    equilibration_control: str = "nose-hoover"
    melting_cycle: bool = False
    cores: Optional[int] = 1

In [9]:
import random
import string

In [10]:
def _generate_random_string(length: str) -> str:
    return ''.join(random.choices(string.ascii_uppercase + string.digits, k=length))

In [11]:
def _prepare_potential_and_structure(potential, structure):
    from pyiron_atomistics.lammps.potential import LammpsPotential, LammpsPotentialFile
    from pyiron_atomistics.lammps.structure import (
        LammpsStructure,
    ) 

    potential_df = LammpsPotentialFile().find_by_name(potential)
    potential = LammpsPotential()
    potential.df = potential_df

    pair_style = []
    pair_coeff = []
    
    pair_style.append(" ".join(potential.df["Config"].to_list()[0][0].strip().split()[1:]))
    pair_coeff.append(" ".join(potential.df["Config"].to_list()[0][1].strip().split()[1:]))

    #now prepare the list of elements
    elements = potential.get_element_lst()
    elements_from_pot = potential.get_element_lst()

    lmp_structure = LammpsStructure()
    lmp_structure.potential = potential
    lmp_structure.atom_type = "atomic"
    lmp_structure.el_eam_lst = potential.get_element_lst()
    lmp_structure.structure = structure

    elements_object_lst = structure.get_species_objects()
    elements_struct_lst = structure.get_species_symbols()

    masses = []
    for element_name in elements_from_pot:
        if element_name in elements_struct_lst:
            index = list(elements_struct_lst).index(element_name)
            masses.append(elements_object_lst[index].AtomicMass)
        else:
            masses.append(1.0)

    file_name = os.path.join(os.getcwd(), _generate_random_string(7)+'.dat')
    lmp_structure.write_file(file_name=file_name)
    potential.copy_pot_files(os.getcwd())
    return pair_style, pair_coeff, elements, masses, file_name

In [12]:
def _prepare_input(inp, potential, structure, mode='fe', reference_phase='solid'):
    from calphy.input import Calculation
    import os
    pair_style, pair_coeff, elements, masses, file_name = _prepare_potential_and_structure(potential, structure)

    inpdict = asdict(inp)
    inpdict["pair_style"] = pair_style
    inpdict["pair_coeff"] = pair_coeff
    inpdict["element"] = elements
    inpdict["mass"] = masses
    inpdict['mode'] = mode
    inpdict['reference_phase'] = reference_phase
    inpdict['lattice'] = file_name
    inpdict["queue"] = {"cores": inpdict["cores"],}
    del inpdict["cores"]

    if inpdict["md"] is None:
        inpdict["md"] = {
                "timestep": 0.001,
                "n_small_steps": 10000,
                "n_every_steps": 10,
                "n_repeat_steps": 10,
                "n_cycles": 100,
                "thermostat_damping": 0.5,
                "barostat_damping": 0.1,
        }
    if inpdict["tolerance"] is None:
        inpdict["tolerance"] = {
                "spring_constant": 0.01,
                "solid_fraction": 0.7,
                "liquid_fraction": 0.05,
                "pressure": 1.0,
        }
    if inpdict["nose_hoover"] is None:
        inpdict["nose_hoover"] = {
                "thermostat_damping": 0.1,
                "barostat_damping": 0.1,
        }
    if inpdict["berendsen"] is None:
        inpdict["berendsen"] = {
                "thermostat_damping": 100.0,
                "barostat_damping": 100.0,
        }
    if mode == 'ts':
        inpdict["temperature"] = [inpdict['temperature'], inpdict["temperature_stop"]]
        del inpdict["temperature_stop"]
        
    calc = Calculation(**inpdict)
    return calc

In [13]:
def _run_cleanup(simfolder, lattice, delete_folder=False):
    import shutil
    import os
    os.remove(lattice)
    if delete_folder:
        shutil.rmtree(simfolder)

In [14]:
from ase.atoms import Atoms

In [15]:
@as_function_node
def SolidFreeEnergy(inp, structure: Atoms, potential: str) -> float:
    """
    Calculate the free energy of a solid phase.

    Parameters:
    -----------
    inp: InputClass
        Input parameters for calphy calculations.
    structure: Atoms
        Atomic structure.
    potential: str
        Potential name.
    
    Returns:
    --------
    float
        Free energy in eV/atom
    """
    from calphy.solid import Solid
    from calphy.routines import routine_fe
    import os

    calc = _prepare_input(inp, potential, structure, mode='fe', reference_phase='solid')
    #os.chdir()
    simfolder = calc.create_folders()
    job = Solid(calculation=calc, simfolder=simfolder)
    job = routine_fe(job)
    _run_cleanup(simfolder, calc.lattice)
    free_energy = job.report["results"]["free_energy"]
    return free_energy

In [16]:
@as_function_node
def LiquidFreeEnergy(inp, structure: Atoms, potential: str) -> float:
    """
    Calculate the free energy of a liquid phase.

    Parameters:
    -----------
    inp: InputClass
        Input parameters for calphy calculations.
    structure: Atoms
        Atomic structure.
    potential: str
        Potential name.
    
    Returns:
    --------
    float
        Free energy in eV/atom
    """
    from calphy.liquid import Liquid
    from calphy.routines import routine_fe
    
    calc = _prepare_input(inp, potential, structure, mode='fe', reference_phase='liquid')
    simfolder = calc.create_folders()
    job = Liquid(calculation=calc, simfolder=simfolder)
    job = routine_fe(job)
    #run calculation
    _run_cleanup(simfolder, calc.lattice)
    free_energy = job.report["results"]["free_energy"]
    return free_energy

In [17]:
import numpy as np
import pandas as pd

In [18]:
from typing import Tuple

In [19]:
@as_function_node
def SolidFreeEnergyWithTemp(inp, structure: Atoms, potential: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Calculate the free energy of a solid phase as a function of temperature.

    Parameters:
    -----------
    inp: InputClass
        Input parameters for calphy calculations.
    structure: Atoms
        Atomic structure.
    potential: str
        Potential name.

    Returns:
    --------
    Tuple[np.ndarray, np.ndarray]
        Temperature and free energy in K and eV/atom, respectively.
    """
    from calphy.solid import Solid
    from calphy.routines import routine_ts
    
    calc = _prepare_input(inp, potential, structure, mode='ts', reference_phase='solid')
    simfolder = calc.create_folders()
    job = Solid(calculation=calc, simfolder=simfolder)
    job = routine_ts(job)
    #run calculation

    #grab the results
    datafile = os.path.join(os.getcwd(), simfolder, 'temperature_sweep.dat')
    temperature, free_energy = np.loadtxt(datafile, unpack=True, usecols=(0,1))

    _run_cleanup(simfolder, calc.lattice)
    return temperature, free_energy

In [20]:
@as_function_node
def LiquidFreeEnergyWithTemp(inp, structure: Atoms, potential: str) -> Tuple[np.ndarray, np.ndarray]:
    """
    Calculate the free energy of a liquid phase as a function of temperature.

    Parameters:
    -----------
    inp: InputClass
        Input parameters for calphy calculations.
    structure: Atoms
        Atomic structure.
    potential: str
        Potential name.

    Returns:
    --------
    Tuple[np.ndarray, np.ndarray]
        Temperature and free energy in K and eV/atom, respectively.
    """
    from calphy.liquid import Liquid
    from calphy.routines import routine_ts
    
    calc = _prepare_input(inp, potential, structure, mode='ts', reference_phase='liquid')
    simfolder = calc.create_folders()
    job = Liquid(calculation=calc, simfolder=simfolder)
    job = routine_ts(job)
    
    #grab the results
    datafile = os.path.join(os.getcwd(), simfolder, 'temperature_sweep.dat')
    temperature, free_energy = np.loadtxt(datafile, unpack=True, usecols=(0,1))

    _run_cleanup(simfolder, calc.lattice)
    return temperature, free_energy

In [21]:
@as_function_node("fig")
def PlotFreeEnergy(temperature: np.ndarray, free_energy: np.ndarray):
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots()
    ax.plot(temperature, free_energy, label='free energy')
    ax.set_ylabel('Free energy (eV/atom)')
    ax.set_xlabel('Temperature (K)')
    plt.legend(frameon=False)
    return fig

In [22]:
@as_function_node
def CalcPhaseTransformationTemp(temp_A: np.ndarray, fe_A: np.ndarray, temp_B: np.ndarray, fe_B: np.ndarray, fit_order: int = 4):
    """
    Calculate the phase transformation temperature from free energy data.

    Parameters:
    -----------
    temp_A: np.ndarray
        Temperature array for phase 1.
    fe_A: np.ndarray
        Free energy array for phase 1.
    temp_B: np.ndarray
        Temperature array for phase 2.
    fe_B: np.ndarray
        Free energy array for phase 2.
    fit_order: int
        Order of the polynomial fit.
    
    Returns:
    --------
    float
        Phase transformation temperature
    """
    import matplotlib.pyplot as plt
    import warnings

    #do some fitting to determine temps
    t1min = np.min(temp_A)
    t2min = np.min(temp_B)
    t1max = np.max(temp_A)
    t2max = np.max(temp_B)

    tmin = np.min([t1min, t2min])
    tmax = np.max([t1max, t2max])

    #warn about extrapolation
    if not t1min == t2min:
        warnings.warn(f'free energy is being extrapolated!')
    if not t1max == t2max:
        warnings.warn(f'free energy is being extrapolated!')

    #now fit
    f1fit = np.polyfit(temp_A, fe_A, fit_order)
    f2fit = np.polyfit(temp_B, fe_B, fit_order)

    #reevaluate over the new range
    fit_t = np.arange(tmin, tmax+1, 1)
    fit_f1 = np.polyval(f1fit, fit_t)
    fit_f2 = np.polyval(f2fit, fit_t)

    #now evaluate the intersection temp
    arg = np.argsort(np.abs(fit_f1-fit_f2))[0]
    phase_transition_temperature = fit_t[arg]

    #warn if the temperature is shady
    if np.abs(transition_temp-tmin) < 1E-3:
        warnings.warn('It is likely there is no intersection of free energies')
    elif np.abs(transition_temp-tmax) < 1E-3:
        warnings.warn('It is likely there is no intersection of free energies')

    #plot
    c1lo = '#ef9a9a'
    c1hi = '#b71c1c'
    c2lo = '#90caf9'
    c2hi = '#0d47a1'

    fig, ax = plt.subplots()
    ax.plot(fit_t, fit_f1, color=c1lo, label=f'phase A fit')
    ax.plot(fit_t, fit_f2, color=c2lo, label=f'phase B fit')
    ax.plot(temp_A, fe_A, color=c1hi, label='phase A', ls='dashed')
    ax.plot(temp_B, fe_B, color=c2hi, label='phase B', ls='dashed')
    ax.axvline(transition_temp, ls='dashed', c='#37474f')
    ax.set_ylabel('Free energy (eV/atom)')
    ax.set_xlabel('Temperature (K)')
    ax.legend(frameon=False)

    return phase_transition_temperature, fig

In [23]:
@as_function_node
def CollectResults() -> pd.DataFrame:
    from calphy.postprocessing import gather_results
    results = gather_results('.')
    return results

In [24]:
import os

In [25]:
from dataclasses import asdict

In [26]:
from pyiron_core.pyiron_nodes.atomistic.structure.build import Bulk

In [27]:
from pyiron_core.pyiron_nodes.atomistic.structure.transform import Repeat

In [28]:
wf = Workflow("calphy-fe1")

In [29]:
wf.Bulk = Bulk(name="Al", cubic=True)
wf.Repeat = Repeat(structure=wf.Bulk, repeat_scalar=3)
wf.InputClass = InputClass()
wf.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(inp=wf.InputClass, structure=wf.Repeat, potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1")
wf.PlotFreeEnergy = PlotFreeEnergy(temperature=wf.SolidFreeEnergyWithTemp.outputs.temperature, free_energy=wf.SolidFreeEnergyWithTemp.outputs.free_energy)

In [30]:
pf = PyironFlow([wf])
pf.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Need the solution?</b></p>
Copy paste the following code into a cell and execute:
</br>
</br>
pf = PyironFlow([Workflow("calphy-fe1")]) </br>
pf.gui

</div>

## <font style="font-family:roboto;color:#455e6c"> Calculate melting temperature </font> 

Now we take a step forward and calculate the melting temperature. The melting temperature is the temperature at which the free energies of the solid and liquid phases are equal. 

In addition to the nodes above, the following would be useful:
- The steps for solid free energy are same as above, but with good temperature window to be set in the `InputClass` would be 800-1100 K.
- For liquid, we need to create a structure first. There are two ways to do it:
    - either check the `melting_cycle` box in `InputClass`
    - or use `atomistic -> structure -> transform -> Rattle` with `stdev` of approximately 0.6 to create a structure, similar to liquid. Hint: Use the `Plot3D` node to see how it looks like!
- `atomistic -> thermodynamics -> calphy -> LiquidFreeEnergyWithTemp` for calculating the free energy of the liquid phase
- A good temperature window to be set in the `InputClass` would be 800-1100 K.

In [44]:
@as_function_node
def Rattle(structure, seed: int = 42, stdev: float = 0.1):
    """
    Randomly displace the atoms in the structure.

    Parameters:

    seed = 42:
    Random seed for the random number generator.
    amplitude = 0.1:
    The amplitude of the random displacement.
    """
    structure = structure.copy()
    structure.rattle(seed=seed, stdev=stdev)
    return structure

In [45]:
@as_function_node
def TemperatureLinePhase(
        name: str,
        concentration: float,
        temperatures: np.ndarray | list[float],
        free_energies: np.ndarray | list[float],
        num_parameters: int | None = 3
):
    phase = landau.phases.TemperatureDependentLinePhase(
            name, concentration, temperatures, free_energies,
            landau.interpolate.SGTE(num_parameters)
    )
    return phase

In [46]:
import seaborn as sns

In [47]:
@as_function_node
def TransitionTemperature(
        phase1, phase2,
        Tmin: int | float,
        Tmax: int | float,
) -> float:
    """Plot free energies of two phases and find their intersection, i.e. the transition temperature.

    Assumes that both phases are of the same concentration, otherwise the results will be off, as it takes the chemical
    potential difference to be zero.

    Args:
        phase1, phase2 (landau.phases.Phase): the two phases to plot
        Tmin (float): minimum temperature
        Tmax (float): maximum temperature

    Returns:
        float: transition temperature if found, else NaN
    """
    import seaborn as sns
    import matplotlib.pyplot as plt
    import numpy as np
    df = landau.calculate.calc_phase_diagram([phase1, phase2], np.linspace(Tmin, Tmax), mu=0.0, keep_unstable=True)
    try:
        fm, Tm = df.query('border and T!=@Tmin and T!=@Tmax')[['f','T']].iloc[0].tolist()
    except IndexError:
        print("Transition Point not found!")
        fm, Tm = np.nan, np.nan
    fig, ax = plt.subplots()
    sns.lineplot(
        data=df,
        x='T', y='f',
        hue='phase',
        style='stable', style_order=[True, False],
        ax=ax,
    )
    ax.axvline(Tm, color='k', linestyle='dotted', alpha=.5)
    ax.scatter(Tm, fm, marker='o', c='k', zorder=10)

    dfa = np.ptp(df['f'].dropna())
    dft = np.ptp(df['T'].dropna())
    ax.text(Tm + .05 * dft, fm + dfa * .1, rf"$T_m = {Tm:.0f}\,\mathrm{{K}}$", rotation='vertical', ha='center')
    ax.set_xlabel("Temperature [K]")
    ax.set_ylabel("Free Energy [eV/atom]")
    return Tm, fig

In [56]:
import landau

In [57]:
wf = Workflow("calphy-tm1")

In [58]:
wf.Bulk = Bulk(name="Al", cubic=True)
wf.Repeat = Repeat(structure=wf.Bulk, repeat_scalar=3)
wf.Rattle = Rattle(structure=wf.Repeat, seed=42, stdev=0.6)
wf.InputClass = InputClass(temperature=800, temperature_stop=1100)
wf.SolidFreeEnergyWithTemp = SolidFreeEnergyWithTemp(inp=wf.InputClass, structure=wf.Repeat, potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1")
wf.LiquidFreeEnergyWithTemp = LiquidFreeEnergyWithTemp(inp=wf.InputClass, structure=wf.Rattle, potential="2009--Mendelev-M-I--Al-Mg--LAMMPS--ipr1")
wf.Solid = TemperatureLinePhase(name="AlSolid", concentration=0, temperatures=wf.SolidFreeEnergyWithTemp.outputs.temperature, free_energies=wf.SolidFreeEnergyWithTemp.outputs.free_energy)
wf.Liquid = TemperatureLinePhase(name="AlLiquid", concentration=0, temperatures=wf.LiquidFreeEnergyWithTemp.outputs.temperature, free_energies=wf.LiquidFreeEnergyWithTemp.outputs.free_energy)
wf.TransitionTemperature = TransitionTemperature(phase1=wf.Solid, phase2=wf.Liquid, Tmin=800, Tmax=1000)

In [59]:
pf = PyironFlow([wf])

In [60]:
pf.gui

<div class="admonition note" name="html-admonition" style="background: #FFEDD1; padding: 10px">
<p class="title"><b>Need the solution?</b></p>
Copy paste the following code into a cell and execute:
</br>
</br>
pf = PyironFlow([Workflow("calphy-tm1")]) </br>
pf.gui

</div>

## <font style="font-family:roboto;color:#455e6c"> Calculating a binary phase diagram </font> 

Now that we have seen the thermodynamic functions in Landau, we will make a simple eutectic binary phase diagram with Landau.

In [63]:
@as_function_node
def LinePhase(name: str, concentration: float, energy: float, entropy: float) -> landau.phases.LinePhase:
    import landau
    phase = landau.phases.LinePhase(name, concentration, energy, entropy)
    return phase

In [64]:
@as_function_node
def TemperatureLinePhase(
        name: str,
        concentration: float,
        temperatures: np.ndarray | list[float],
        free_energies: np.ndarray | list[float],
        num_parameters: int | None = 3
):
    phase = landau.phases.TemperatureDependentLinePhase(
            name, concentration, temperatures, free_energies,
            landau.interpolate.SGTE(num_parameters)
    )
    return phase

In [65]:
@as_function_node
def IdealSolution(name: str, phase1: landau.phases.Phase, phase2: landau.phases.Phase) -> landau.phases.Phase:
    import landau
    phase = landau.phases.IdealSolution(name, phase1, phase2)
    return phase

In [67]:
@as_function_node
def List5(x1, x2=None, x3=None, x4=None, x5=None) -> list:
    list_out = [x for x in (x1, x2, x3, x4, x5) if x is not None]
    return list_out

In [69]:
def guess_mu_range(phases, Tmax, samples):
    """Guess chemical potential window from the ideal solution.

    Searches numerically for chemical potentials which stabilize
    concentrations close to 0 and 1 and then use the concentrations
    encountered along the way to numerically invert the c(mu) mapping.
    Using an even c grid with mu(c) then yields a decent sampling of mu
    space so that the final phase diagram is described everywhere equally.

    Args:
        phases: list of phases to consider
        Tmax: temperature at which to estimate 
        samples: how many mu samples to return

    Returns:
        array of chemical potentials that likely cover the whole concentration space
    """

    import scipy.optimize as so
    import scipy.interpolate as si
    import numpy as np
    # semigrand canonical "average" concentration
    # use this to avoid discontinuities and be phase agnostic
    def c(mu):
        phis = np.array([p.semigrand_potential(Tmax, mu) for p in phases])
        conc = np.array([p.concentration(Tmax, mu) for p in phases])
        phis -= phis.min()
        beta = 1/(Tmax*8.6e-5)
        prob = np.exp(-beta*phis)
        prob /= prob.sum()
        return (prob * conc).sum()
    cc, mm = [], []
    mu0, mu1 = 0, 0
    while (ci := c(mu0)) > 0.001:
        cc.append(ci)
        mm.append(mu0)
        mu0 -= 0.05
    while (ci := c(mu1)) < 0.999:
        cc.append(ci)
        mm.append(mu1)
        mu1 += 0.05
    cc = np.array(cc)
    mm = np.array(mm)
    I = cc.argsort()
    cc = cc[I]
    mm = mm[I]
    return si.interp1d(cc, mm)(np.linspace(min(cc), max(cc), samples))

In [71]:
@as_function_node
def CalcPhaseDiagram(
        phases: list,
        temperatures: list[float] | np.ndarray,
        chemical_potentials: list[float] | np.ndarray | int = 100,
        refine: bool = True
):
    """Calculate thermodynamic potentials and respective stable phases in a range of temperatures.

    The chemical potential range is chosen automatically to cover the full concentration space.

    Args:
        phases: list of phases to consider
        temperatures: temperature samples
        mu_samples: number of samples in chemical potential space
        refine (bool): add additional sampling points along exact phase transitions

    Returns:
        dataframe with phase data
    """
    import matplotlib.pyplot as plt
    import landau

    if isinstance(chemical_potentials, int):
        mus = guess_mu_range(phases, max(temperatures), chemical_potentials)
    else:
        mus = chemical_potentials
    phase_data = landau.calculate.calc_phase_diagram(
            phases, np.asarray(temperatures), mus,
            refine=refine, keep_unstable=False
    )
    return phase_data

In [87]:
@as_function_node
def PlotConcPhaseDiagram(
        phase_data,
        plot_samples: bool = False,
        plot_isolines: bool = False,
        plot_tielines: bool = True,
        linephase_width: float = 0.01,
        concavity: float | None = None,
):
    """Plot a concentration-temperature phase diagram.

    phase_data should originate from CalcPhaseDiagram.

    Args:
        phases: list of phases to consider
        plot_samples (bool): overlay points where phase data has been sampled
        plot_isolines (bool): overlay lines of constance chemical potential
        plot_tielines (bool): add grey lines connecting triple points
        linephase_width (float): phases that have a solubility less than this
            will be plotted as a rectangle
        concavity (float, optional, range in [0, 1]): how aggressive to be when
            fitting polyhedra to samples phase data; lower means more ragged
            shapes, higher means smoother; 1 corresponds to convex hull of points
    """
    import matplotlib.pyplot as plt
    import seaborn as sns
    import landau
    
    fig, ax = plt.subplots()
    landau.plot.plot_phase_diagram(
            df=phase_data.drop('refined', errors='ignore', axis='columns'),
            min_c_width=linephase_width,
            alpha=concavity or 0.1,
    )
    if plot_samples:
        sns.scatterplot(
            data=phase_data,
            x='c', y='T',
            hue='phase',
            legend=False,
            s=1
        )
    if plot_isolines:
        sns.lineplot(
            data=phase_data.loc[np.isfinite(phase_data.mu)],
            x='c', y='T',
            hue='mu',
            units='phase', estimator=None,
            legend=False,
            sort=False,
        )
    if plot_tielines and 'refined' in phase_data.columns:
        # hasn't made it upstream yet
        for T, dd in phase_data.query('refined=="delaunay-triple"').groupby('T'):
            plt.plot(dd.c, [T]*3, c='k', alpha=.5, zorder=-10)
    plt.xlabel("Concentration")
    plt.ylabel("Temperature [K]")
    return fig


In [88]:
from pyiron_core.pyiron_nodes.math import Linspace

In [109]:
wf = Workflow("landau-toy")

In [110]:
wf.LinePhase0 = LinePhase(name="SolidA", concentration=0, energy=2.45, entropy=0.00001)
wf.LinePhase1 = LinePhase(name="SolidB", concentration=1, energy=2.95, entropy=0.00001)
wf.LinePhase2 = LinePhase(name="LiquidA", concentration=0, energy=2.5, entropy=0.0001)
wf.LinePhase3 = LinePhase(name="LiquidB", concentration=1, energy=3.1, entropy=0.0002)
wf.IdealSolution = IdealSolution(name="liquid", phase1=wf.LinePhase2, phase2=wf.LinePhase3)
wf.Linspace = Linspace(x_min=250, x_max=1000, num_points=25)
wf.List5 = List5(x1=wf.LinePhase0, x2=wf.LinePhase1, x3=wf.IdealSolution)
wf.CalcPhaseDiagram = CalcPhaseDiagram(phases=wf.List5, temperatures=wf.Linspace, chemical_potentials=50)
wf.PlotConcPhaseDiagram = PlotConcPhaseDiagram(phase_data=wf.CalcPhaseDiagram, plot_tielines=True)

In [111]:
pf = PyironFlow([wf])
pf.gui

## <font style="font-family:roboto;color:#455e6c"> MgCa phase diagram </font> 

Now we will take the same approach to calculate a phase diagram with real data, in this case the MgCa system. The data can be found [here](https://github.com/eisenforschung/mgalca-mtp-data).

In [ ]:
pf = PyironFlow([Workflow("landau-MgCa")])
pf.gui

---
---

### <font style="color:#455e6c" face="Helvetica" > Calculation of free energies: Thermodynamic integration </font>

<img src="img/fig1.png" width="1000">

- free energy of reference system are known: Einstein crystal, [Uhlenbeck-Ford model](https://doi.org/10.1063/1.4967775)
- the two systems are coupled by 
$$
H(\lambda) = \lambda H_f + (1-\lambda)\lambda H_i
$$
- Run calculations for each $\lambda$ and integrate 
$$
G_f = G_i + \int_{\lambda=0}^1 d\lambda \bigg \langle  \frac{\partial H(\lambda)}{\partial \lambda } \bigg \rangle
$$

To calculate $F$,

- for each phase
    - for each pressure
        - for each temperature
            - for each $\lambda$

If we choose 100 different $\lambda$ values; 100 calculations are needed for each temperature and pressure! 

**Dimensionality: (phase, $P$, $T$, $\lambda$)**

### <font style="color:#455e6c" face="Helvetica" > Speeding things up: Non-equilibrium calculations </font>

##### Non-Equilibrium Hamiltonian Interpolation

<img src="img/cimg5.png" width="600">

In this method:

- Discrete $\lambda$ parameter is replaced by a time dependent $\lambda(t)$
- Instead of running calculations at each $\lambda$, run a single, short, non-equilibrium calculation

$$
G_f = G_i + \int_{t_i}^{t_f} dt \frac{d\lambda (t)}{dt}  \frac{ H(\lambda)}{\partial \lambda }
$$

As discussed:
- the coupling parameter $\lambda$ earlier is replaced by a time dependent parameter
- The equation also no longer has an ensemble average  

These aspects makes it quite easy and fast to estimate this integral.

However:
- this equation holds when the switching betwen the system of interest and reference system is carried out infinitely slowly
- Practically, this is not not possible. 

Therefore we can write:

$$
\Delta G = W_{rev} = W_s - E_d
$$

$$
W_s = \int_{t_i}^{t_f} dt \frac{d\lambda (t)}{dt}  \frac{ H(\lambda)}{\partial \lambda }
$$

- $E_d$ is the energy dissipation
- $E_d \to 0$ when $t_f-t_i \to \infty$

So far, so good, but how is this useful?

- Instead of a single transformation from system of interest to reference, we switch back too
- These are called forward $(i \to f)$ and backward $(f \to i)$ switching
- $t_f - t_i = t_{sw}$ is the switching time in each direction
- If $t_{sw}$ is long enough, $E_d^{i \to f} = E_d^{f \to i}$
- and $\Delta G = \frac{1}{2} (W_s^{i \to f} - W_s^{f \to i})$

Now, we have all the components required for actual calculations.

We have also managed to successfully reduce the dimensionality

- for each phase
    - for each pressure
        - for each temperature
            - ~~for each $\lambda$~~

**Dimensionality: (phase, $P$, $T$)**


So, how do we calculate the free energy of a system modelled with a given interatomic potential?

### <font style="color:#455e6c" face="Helvetica" > Free energy for solids </font>

#### Task: Find free energy of Al in FCC lattice at 500 K and 0 pressure

1. Create an Al FCC lattice
2. Choose an interatomic potential
3. Run MD calculations at 500 K and 0 pressure to equilibrate the system
4. Introduce the reference system
5. Switch....
6. .....

Steps 1-3 should be fairly easy, we saw this in the last days and also in the first session. But how do we introduce a reference system?

- A reference system is simply one for which the free energy is analytically known ($G_i$)
- We calculate the free energy difference between this and the system of interest.

In case of solids, a good choice of a reference system is the Einstein crystal. An Einstein crystal is a set of independent harmonic oscillators attached to the lattice positions. 


The free energy of the Einstein crystal is:

$$
F_E = 3 k_B T \sum_{i} ln \bigg ( \frac{\hbar \omega_i}{k_B T} \bigg )
$$

We need to calculate:

- $\omega$
- A common way is $$  \frac{1}{2} k_i \langle (\Delta \pmb{r}_i)^2 \rangle = \frac{3}{2} k_\mathrm{B} T $$
- $\langle (\Delta \pmb{r}_i)^2 \rangle$ is the mean squared displacement.

Now that we know about the reference system, let's update our pseudo workflow:

1. Create an Al fcc lattice
2. Choose an interatomic potential
3. Run MD calculations at 500 K and 0 pressure to equilibrate the system
4. Calculate the mean squared displacement, therefore spring constants
5. Switch system of interest to reference system
6. Equilibrate the system
7. Switch back to system of interest
8. Find the work done
9. Add to the free energy of the Einstein crystal

As you can see, there are a number of steps that need to be done. This is where **calphy** and **pyiron** come in. These tools automatise all of the above steps and makes it easy to perform these calculations.

### <font style="color:#455e6c" face="Helvetica" > Free energy as function of temperature </font>

<img src="img/cimg6.png" width="600">

Gibb's free energy via reversible scaling at a constant pressure is given by,

$ G(N,P,T_f) = G(N,P,T_i) + \dfrac{3}{3}Nk_BT_f\ln{\dfrac{T_f}{T_i}} + \dfrac{T_f}{T_i}\Delta G $,

Therefore, $G(N,P,T_f)$ can be computed from $G(N,P,T_i)$ via the free energy difference $\Delta G$. 

Here, $\Delta G = \dfrac{1}{2}[W_{if}-W_{fi}$] --- (2)

The reversible work is related to the internal energy $U$ by,
$W = \int_{1}^{\lambda_f}<U> \,d\lambda$ --- (3)

Using MD $W$ can be computed as:
- equilibrate for time $t_{eq}$ in NPT ensemble
- switch $\lambda$ : $1->\dfrac{T_f}{T_i}$ over time $t_{sw}$
- calculate work $W_{if}$ from (3)
- equilibrate for time $t_{eq}$ in NPT ensemble
- switch $\lambda$ : $\dfrac{T_f}{T_i}->1$ over time $t_{sw}$
- calculate work $W_{fi}$ from (3).

### <font style="color:#455e6c" face="Helvetica" > Free energy for liquids </font>

**How is the liquid prepared in this calculation?**

- Start from the given structure
- This structure is heated until it melts.
- Melting of the structure is automatically detected by calphy
- Once melted, it is equilibrated to the required temperature and pressure.

**What about the reference system for liquid?**

The reference system for the Liquid structure is also different. In this case, the Uhlenbeck-Ford system is used as the reference system for liquid.

The Uhlenbeck-Ford model is described by,

$$
E = - \epsilon \log(1-\exp(-r^2/\sigma^2))
$$

where,

$$
\epsilon = p k_B T
$$

$\epsilon$ and $\sigma$ are energy and length scales, respectively.

It is purely repulsive liquid model which does not undergo any phase transformations.

### <font style="color:#455e6c" face="Helvetica" > Comparison of melting temperature methods </font>

<img src="img/tm_methods.png" width="900">

### <font style="color:#455e6c" face="Helvetica" > Further reading </font>
- [Menon et al. "From electrons to phase diagrams with machine learning potentials using pyiron based automated workflows." npj Computational Materials 10, 2024](https://www.nature.com/articles/s41524-024-01441-0)
- [Menon, Sarath, Yury Lysogorskiy, Jutta Rogal, and Ralf Drautz. “Automated Free-Energy Calculation from Atomistic Simulations.” Physical Review Materials 5, no. 10 (October 11, 2021): 103801.](https://doi.org/10.1103/PhysRevMaterials.5.103801).
- [Freitas, Rodrigo, Mark Asta, and Maurice de Koning. “Nonequilibrium Free-Energy Calculation of Solids Using LAMMPS.” Computational Materials Science 112 (February 2016): 333–41.](https://doi.org/10.1016/j.commatsci.2015.10.050).
- [Paula Leite, Rodolfo, and Maurice de Koning. “Nonequilibrium Free-Energy Calculations of Fluids Using LAMMPS.” Computational Materials Science 159 (March 2019): 316–26.](https://doi.org/10.1016/j.commatsci.2018.12.029).
- [Koning, Maurice de, A. Antonelli, and Sidney Yip. “Optimized Free-Energy Evaluation Using a Single Reversible-Scaling Simulation.” Physical Review Letters 83, no. 20 (November 15, 1999): 3973–77.](https://doi.org/10.1103/PhysRevLett.83.3973).
- [Paula Leite, Rodolfo, Rodrigo Freitas, Rodolfo Azevedo, and Maurice de Koning. “The Uhlenbeck-Ford Model: Exact Virial Coefficients and Application as a Reference System in Fluid-Phase Free-Energy Calculations.” The Journal of Chemical Physics 145, no. 19 (November 21, 2016): 194101.](https://doi.org/10.1063/1.4967775).



### <font style="font-family:roboto;color:#455e6c"> Software used in this notebook </font>  

- [calphy](https://calphy.org)
- [landau](https://github.com/eisenforschung/landau)
- [LAMMPS](https://www.lammps.org/)
- [pyiron](https://pyiron.org/)
- [pyiron_workflow](https://github.com/pyiron/pyiron_workflow)

<img src="img/logo_roll.png" width="1200">